# Nifty 50 — Open-to-Close Returns

Brief notebook: load cleaned OHLCV data, compute open-to-close simple & log returns, validate, save.

## 1. Load and sort data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('cleaned_data.csv')
df['Date'] = pd.to_datetime(df['Date'])
df = df.sort_values('Date').reset_index(drop=True)

df.head()

,Date,Close,High,Low,Open,Volume
0,2014-01-02,6221.149902,6358.299805,6211.299805,6301.250000,158100
1,2014-01-03,6211.149902,6221.700195,6171.250000,6194.549805,139000
2,2014-01-06,6191.450195,6224.700195,6170.250000,6220.850098,118300
3,2014-01-07,6162.250000,6221.500000,6144.750000,6203.899902,138600
4,2014-01-08,6174.600098,6192.100098,6160.350098,6178.049805,146900


## 2. Compute open-to-close returns

- **Simple return**: `(Close / Open) - 1`
- **Log return**: `ln(Close / Open)`

Both use only same-day values, so no NaN rows are produced.

In [2]:
df['simple_return_o2c'] = (df['Close'] / df['Open']) - 1
df['log_return_o2c'] = np.log(df['Close'] / df['Open'])

df[['Date', 'Open', 'Close', 'simple_return_o2c', 'log_return_o2c']].head()

,Date,Open,Close,simple_return_o2c,log_return_o2c
0,2014-01-02,6301.250000,6221.149902,-0.012712,-0.012793
1,2014-01-03,6194.549805,6211.149902,0.002680,0.002676
2,2014-01-06,6220.850098,6191.450195,-0.004726,-0.004737
3,2014-01-07,6203.899902,6162.250000,-0.006714,-0.006736
4,2014-01-08,6178.049805,6174.600098,-0.000558,-0.000559


## 3. Validate

Check the mathematical identity `log_return = ln(1 + simple_return)` and confirm no missing values.

In [3]:
assert np.allclose(df['log_return_o2c'], np.log(1 + df['simple_return_o2c']), atol=1e-8)
assert df[['simple_return_o2c', 'log_return_o2c']].isna().sum().sum() == 0

print("Validation passed.")
print(df[['simple_return_o2c', 'log_return_o2c']].describe())

Validation passed.
       simple_return_o2c  log_return_o2c
count        2670.000000     2670.000000
mean           -0.000705       -0.000741
std             0.008491        0.008486
min            -0.068180       -0.070616
25%            -0.004821       -0.004833
50%            -0.000511       -0.000511
75%             0.003691        0.003684
max             0.093065        0.088986


## 4. Save output

In [4]:
df.to_csv('cleaned_data_with_returns.csv', index=False)
print(df.shape)

(2670, 8)
